In [ ]:
# =============================================================
# CELL 1 — Run this first, then RESTART the runtime.
# Runtime → Restart runtime  (Ctrl+M .)  before Cell 2.
# Do NOT skip the restart — unsloth patches torch at import
# time and the patches won't apply to an already-loaded torch.
# =============================================================

import subprocess, sys

# Guard: must be in Colab
try:
    import google.colab  # noqa: F401
except ImportError:
    raise RuntimeError("This notebook must be run in Google Colab.")

print("Installing unsloth + xformers (this takes ~2 min)…")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "xformers",
])

print()
print("=" * 60)
print("  Installation complete.")
print("  *** RESTART THE RUNTIME NOW ***")
print("  Runtime → Restart runtime  (or Ctrl+M .)")
print("  Then run Cell 2.  Do NOT re-run Cell 1.")
print("=" * 60)

In [ ]:
# =============================================================
# CELL 2 — Run AFTER restarting the runtime.
# Before running:
#   1. Upload synthetic_qa.jsonl to Colab Files (/content/).
#   2. Confirm runtime type is GPU  (Runtime → Change runtime type).
# This cell trains the model and saves the adapter to /content/adapter.
# =============================================================

# ── 0. Order-of-execution guard ──────────────────────────────
try:
    from unsloth import FastLanguageModel
except ImportError:
    raise RuntimeError(
        "unsloth is not importable. Did you restart the runtime after Cell 1? "
        "Runtime → Restart runtime, then run Cell 2."
    )

import json, os
from pathlib import Path

DATA_PATH   = Path("/content/synthetic_qa.jsonl")
ADAPTER_DIR = "/content/adapter"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} not found. "
        "Upload synthetic_qa.jsonl via the Files panel on the left."
    )

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Runtime → Change runtime type → T4 or A100."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}  "
      f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)\n")

# ── 1. Load base model in 4-bit ──────────────────────────────
MAX_SEQ_LEN = 1024
MODEL_ID    = "unsloth/Qwen2.5-7B-Instruct"

print(f"Loading {MODEL_ID}…")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,       # auto-detect: bf16 on Ampere+, fp16 otherwise
    load_in_4bit   = True,
)

# ── 2. Apply LoRA ────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r                   = 16,
    lora_alpha          = 32,
    target_modules      = ["q_proj", "v_proj"],
    lora_dropout        = 0.05,
    bias                = "none",
    use_gradient_checkpointing = "unsloth",  # saves ~30% VRAM vs standard
    random_state        = 42,
)
model.print_trainable_parameters()

# ── 3. Load & format data ────────────────────────────────────
SYSTEM_PROMPT = (
    "You are lectureOS, an AI teaching assistant. "
    "Answer student questions accurately and concisely based on lecture material."
)

raw = [json.loads(l) for l in DATA_PATH.read_text().splitlines() if l.strip()]
print(f"\nLoaded {len(raw):,} training examples.")

def to_text(ex: dict) -> str:
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": ex["question"]},
        {"role": "assistant", "content": ex["answer"]},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

from datasets import Dataset
dataset = Dataset.from_list([{"text": to_text(ex)} for ex in raw])
split   = dataset.train_test_split(test_size=0.05, seed=42)
print(f"Train: {len(split['train']):,}  |  Eval: {len(split['test']):,}\n")

# ── 4. Train ─────────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model            = model,
    tokenizer        = tokenizer,
    train_dataset    = split["train"],
    eval_dataset     = split["test"],
    dataset_text_field = "text",
    max_seq_length   = MAX_SEQ_LEN,
    dataset_num_proc = 2,
    args = TrainingArguments(
        output_dir                  = ADAPTER_DIR,
        num_train_epochs            = 2,
        per_device_train_batch_size = 4,
        per_device_eval_batch_size  = 4,
        gradient_accumulation_steps = 4,       # effective batch = 16
        warmup_ratio                = 0.03,
        learning_rate               = 2e-4,
        lr_scheduler_type           = "cosine",
        weight_decay                = 0.01,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 25,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        optim                       = "adamw_8bit",
        report_to                   = "none",
        seed                        = 42,
    ),
)

print("Training…")
trainer_stats = trainer.train()

# ── 5. Save ──────────────────────────────────────────────────
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

train_loss = trainer_stats.training_loss
runtime_min = trainer_stats.metrics["train_runtime"] / 60
print(f"\nTrain loss : {train_loss:.4f}")
print(f"Runtime    : {runtime_min:.1f} min")
print(f"Adapter    : {ADAPTER_DIR}")
print()
print("=" * 60)
print("  DONE — run Cell 3 to download the adapter.")
print("=" * 60)

In [ ]:
# =============================================================
# CELL 3 — Run AFTER Cell 2 completes.
# Zips /content/adapter and triggers a browser download.
# A save dialog will appear — choose where to save adapter.zip.
# =============================================================

import shutil
from pathlib import Path
from google.colab import files

ADAPTER_DIR = Path("/content/adapter")
ZIP_PATH    = Path("/content/adapter.zip")

# Guard: Cell 2 must have completed successfully
if not ADAPTER_DIR.exists():
    raise RuntimeError(
        "/content/adapter not found. "
        "Run Cell 2 first and wait for the DONE message."
    )

required = {"adapter_config.json", "adapter_model.safetensors"}
present  = {p.name for p in ADAPTER_DIR.iterdir()}
missing  = required - present
if missing:
    raise RuntimeError(
        f"Adapter directory looks incomplete — missing: {missing}. "
        "Re-run Cell 2."
    )

print(f"Adapter contents: {sorted(present)}")
print("Zipping…")
shutil.make_archive("/content/adapter", "zip", "/content", "adapter")
size_mb = ZIP_PATH.stat().st_size / 1e6
print(f"adapter.zip  {size_mb:.1f} MB")
print("Downloading…")
files.download(str(ZIP_PATH))